In [5]:
import arcpy
arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False
base = "https://miciudad.barranquilla.gov.co/gis/rest/services/catastro/datosabiertos/FeatureServer/"
def exportar(capa, filtro, salida, tabla=False, paso=10000, maximo=700000):
    url = base + str(capa)
    partes = []
    for ini in range(0, maximo, paso):
        sql = f"({filtro}) AND objectid > {ini} AND objectid <= {ini+paso}"
        vista = arcpy.management.MakeTableView(url, "v", sql) if tabla else arcpy.management.MakeFeatureLayer(url, "v", sql)
        esperado = int(arcpy.management.GetCount(vista)[0])
        arcpy.management.Delete("v")
        if esperado == 0:
            continue
        nombre = f"tmp_{salida}_{ini//1000:03d}"
        for intento in range(3):
            if tabla:
                arcpy.conversion.ExportTable(url, nombre, sql)
            else:
                arcpy.conversion.ExportFeatures(url, nombre, sql)
            if int(arcpy.management.GetCount(nombre)[0]) == esperado:
                break
        partes.append(nombre)
    arcpy.management.Merge(partes, salida)
    for p in partes:
        arcpy.management.Delete(p)
    print(salida, "TOTAL", arcpy.management.GetCount(salida)[0])
exportar(320, "1=1", "Manzana_2026")
exportar(500, "estado = 'Activo'", "Predio_2026", tabla=True)
exportar(601, "1=1", "Rel_Terreno_Predio", tabla=True)
exportar(305, "estado_unidad_construccion = 1", "UnidadConst_2026")

<class 'arcgisscripting.ExecuteError'>: ERROR 003996: Too Many Requests.
Failed to execute (ExportFeatures).


In [6]:
import arcpy, time
arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False
base = "https://miciudad.barranquilla.gov.co/gis/rest/services/catastro/datosabiertos/FeatureServer/"
def pedir(func, *args):
    for intento in range(6):
        try:
            r = func(*args)
            time.sleep(3)
            return r
        except Exception as e:
            print("  servidor ocupado, espero", 60*(intento+1), "s")
            time.sleep(60*(intento+1))
    raise RuntimeError("El servidor no responde, vuelva a ejecutar más tarde")
def exportar(capa, filtro, salida, tabla=False, paso=10000, maximo=700000):
    url = base + str(capa)
    partes = []
    for ini in range(0, maximo, paso):
        sql = f"({filtro}) AND objectid > {ini} AND objectid <= {ini+paso}"
        crear = arcpy.management.MakeTableView if tabla else arcpy.management.MakeFeatureLayer
        vista = pedir(crear, url, "v", sql)
        esperado = int(arcpy.management.GetCount(vista)[0])
        arcpy.management.Delete("v")
        if esperado == 0:
            continue
        nombre = f"tmp_{salida}_{ini//1000:03d}"
        partes.append(nombre)
        if arcpy.Exists(nombre) and int(arcpy.management.GetCount(nombre)[0]) == esperado:
            continue
        for intento in range(3):
            pedir(arcpy.conversion.ExportTable if tabla else arcpy.conversion.ExportFeatures, url, nombre, sql)
            if int(arcpy.management.GetCount(nombre)[0]) == esperado:
                break
        print(" ", nombre, esperado, "→", arcpy.management.GetCount(nombre)[0])
    arcpy.management.Merge(partes, salida)
    for p in partes:
        arcpy.management.Delete(p)
    print(salida, "TOTAL", arcpy.management.GetCount(salida)[0])
exportar(320, "1=1", "Manzana_2026")
exportar(500, "estado = 'Activo'", "Predio_2026", tabla=True)
exportar(601, "1=1", "Rel_Terreno_Predio", tabla=True)
exportar(305, "estado_unidad_construccion = 1", "UnidadConst_2026")

Manzana_2026 TOTAL 8292
  tmp_Predio_2026_000 9952 → 9952
  tmp_Predio_2026_010 9968 → 9968
  tmp_Predio_2026_020 9961 → 9961
  tmp_Predio_2026_030 9940 → 9940
  tmp_Predio_2026_040 9952 → 9952
  tmp_Predio_2026_050 9957 → 9957
  tmp_Predio_2026_060 9962 → 9962
  tmp_Predio_2026_070 9967 → 9967
  tmp_Predio_2026_080 9972 → 9972
  tmp_Predio_2026_090 9965 → 9965
  tmp_Predio_2026_100 9984 → 9984
  tmp_Predio_2026_110 9989 → 9989
  tmp_Predio_2026_120 9805 → 9805
  tmp_Predio_2026_130 9628 → 9628
  tmp_Predio_2026_140 9993 → 9993
  tmp_Predio_2026_150 9983 → 9983
  tmp_Predio_2026_160 9765 → 9765
  tmp_Predio_2026_170 9412 → 9412
  tmp_Predio_2026_180 9915 → 9915
  tmp_Predio_2026_190 9982 → 9982
  tmp_Predio_2026_200 9990 → 9990
  tmp_Predio_2026_210 10000 → 10000
  tmp_Predio_2026_220 9957 → 9957
  tmp_Predio_2026_230 9984 → 9984
  tmp_Predio_2026_240 9998 → 9998
  tmp_Predio_2026_250 9996 → 9996
  tmp_Predio_2026_260 9973 → 9973
  tmp_Predio_2026_270 9957 → 9957
  tmp_Predio_2026_280 

<class 'RuntimeError'>: El servidor no responde, vuelva a ejecutar más tarde

In [7]:
d = arcpy.Describe(base + "601")
print("OID:", d.OIDFieldName)
print([f.name for f in d.fields])
try:
    arcpy.management.MakeTableView(base + "601", "v")
    print("OK", arcpy.management.GetCount("v")[0])
except Exception as e:
    print("ERROR:", e)


<class 'AttributeError'>: DescribeData: Method OIDFieldName does not exist

In [ ]:
import urllib.request, urllib.parse, json, csv, time, os
url = base + "601/query"
filas, offset = [], 0
while True:
    q = urllib.parse.urlencode({"where": "1=1", "outFields": "*", "resultOffset": offset,
                                "resultRecordCount": 2000, "f": "json"})
    for intento in range(6):
        try:
            data = json.load(urllib.request.urlopen(url + "?" + q, timeout=120))
            break
        except Exception as e:
            print("espero", 60*(intento+1), "s:", e)
            time.sleep(60*(intento+1))
    if "error" in data:
        print("ERROR:", data["error"]); break
    feats = data.get("features", [])
    filas += [f["attributes"] for f in feats]
    if len(filas) % 20000 < 2000:
        print("van", len(filas))
    if not feats or not data.get("exceededTransferLimit"):
        break
    offset += len(feats)
    time.sleep(2)
carpeta = arcpy.mp.ArcGISProject("CURRENT").homeFolder
ruta = os.path.join(carpeta, "rel_terreno_predio.csv")
with open(ruta, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(filas[0].keys()))
    w.writeheader(); w.writerows(filas)
arcpy.conversion.ExportTable(ruta, "Rel_Terreno_Predio")
print("TOTAL", arcpy.management.GetCount("Rel_Terreno_Predio")[0], "campos:", list(filas[0].keys()))

espero 60 s: HTTP Error 403: Forbidden


In [1]:
import arcpy, urllib.request, urllib.parse, urllib.error, json, csv, time, os
arcpy.env.overwriteOutput = True
url = "https://miciudad.barranquilla.gov.co/gis/rest/services/catastro/datosabiertos/FeatureServer/601/query"
filas, offset = [], 0
while True:
    q = urllib.parse.urlencode({"where": "1=1", "outFields": "*", "resultOffset": offset,
                                "resultRecordCount": 2000, "f": "json"})
    req = urllib.request.Request(url + "?" + q, headers={"User-Agent": "Mozilla/5.0"})
    data = None
    for intento in range(3):
        try:
            data = json.load(urllib.request.urlopen(req, timeout=120))
            break
        except urllib.error.HTTPError as e:
            print("HTTP", e.code)
            if e.code == 403:
                break
            time.sleep(30)
        except Exception as e:
            print("error de red:", e); time.sleep(30)
    if data is None or "error" in data:
        print("SE DETIENE:", data and data.get("error")); break
    feats = data.get("features", [])
    filas += [f["attributes"] for f in feats]
    if len(filas) % 20000 < 2000:
        print("van", len(filas))
    if not feats or not data.get("exceededTransferLimit"):
        break
    offset += len(feats)
    time.sleep(2)
if filas:
    ruta = os.path.join(arcpy.mp.ArcGISProject("CURRENT").homeFolder, "rel_terreno_predio.csv")
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(filas[0].keys()))
        w.writeheader(); w.writerows(filas)
    arcpy.conversion.ExportTable(ruta, "Rel_Terreno_Predio")
    print("TOTAL", arcpy.management.GetCount("Rel_Terreno_Predio")[0], "campos:", list(filas[0].keys()))

HTTP 403
SE DETIENE: None


In [2]:
import arcpy
arcpy.env.overwriteOutput = True
for fc, campo in [("Terreno_2026", "Name"), ("Predio_2026", "numero_predial_nacional")]:
    if "clave21" not in [f.name for f in arcpy.ListFields(fc)]:
        arcpy.management.AddField(fc, "clave21", "TEXT", field_length=21)
    arcpy.management.CalculateField(fc, "clave21", f"!{campo}![:21] if !{campo}! else None", "PYTHON3")
    print(fc, "clave21 lista")
arcpy.management.FeatureToPoint("Terreno_2026", "Terreno_pts", "INSIDE")
lyr = arcpy.management.MakeFeatureLayer("Terreno_pts", "pts_lyr")
arcpy.management.AddJoin(lyr, "clave21", "Predio_2026", "clave21", "KEEP_COMMON",
                         join_operation="JOIN_ONE_TO_MANY")
arcpy.conversion.ExportFeatures(lyr, "Predio_pts_2026")
print("Predios ubicados:", arcpy.management.GetCount("Predio_pts_2026")[0],
      "de", arcpy.management.GetCount("Predio_2026")[0])

Terreno_2026 clave21 lista
Predio_2026 clave21 lista
Predios ubicados: 525450 de 475917


In [3]:
import arcpy
arcpy.env.overwriteOutput = True
arcpy.management.FeatureToPoint("Terreno_2026", "Terreno_pts", "INSIDE")
n0 = int(arcpy.management.GetCount("Terreno_pts")[0])
arcpy.management.DeleteIdentical("Terreno_pts", ["clave21"])
n1 = int(arcpy.management.GetCount("Terreno_pts")[0])
print("Puntos de terreno:", n0, "→ claves únicas:", n1, "(repetidos:", n0 - n1, ")")
lyr = arcpy.management.MakeFeatureLayer("Terreno_pts", "pts_lyr2")
arcpy.management.AddJoin(lyr, "clave21", "Predio_2026", "clave21", "KEEP_COMMON",
                         join_operation="JOIN_ONE_TO_MANY")
arcpy.conversion.ExportFeatures(lyr, "Predio_pts_2026")
n = int(arcpy.management.GetCount("Predio_pts_2026")[0])
total = int(arcpy.management.GetCount("Predio_2026")[0])
print("Predios ubicados:", n, "de", total, f"({n/total:.1%})", "| sin ubicar:", total - n)

Puntos de terreno: 206578 → claves únicas: 205454 (repetidos: 1124 )
Predios ubicados: 475129 de 475917 (99.8%) | sin ubicar: 788


In [4]:
import arcpy
arcpy.env.overwriteOutput = True
arcpy.env.outputCoordinateSystem = arcpy.SpatialReference(9377)
ago = "https://services3.arcgis.com/oGYAc07w6wsvgUYr/arcgis/rest/services/"
arcpy.conversion.ExportFeatures(ago + "Localidades_Barrios/FeatureServer/1", "Barrios")
arcpy.conversion.ExportFeatures(ago + "ZHGU_Catastro/FeatureServer/0", "ZHGU_2025")
print("Barrios:", arcpy.management.GetCount("Barrios")[0], "| ZHGU:", arcpy.management.GetCount("ZHGU_2025")[0])
arcpy.analysis.SpatialJoin("Predio_pts_2026", "Barrios", "tmp_pb", "JOIN_ONE_TO_ONE", "KEEP_ALL", match_option="INTERSECT")
arcpy.analysis.SpatialJoin("tmp_pb", "ZHGU_2025", "Predio_analisis_2026", "JOIN_ONE_TO_ONE", "KEEP_ALL", match_option="INTERSECT")
arcpy.management.Delete("tmp_pb")
print("Predio_analisis_2026:", arcpy.management.GetCount("Predio_analisis_2026")[0])

Barrios: 204 | ZHGU: 1467
Predio_analisis_2026: 475129


In [5]:
import arcpy
campos = [f.name for f in arcpy.ListFields("Predio_analisis_2026")]
print(len(campos), "campos:")
print(campos)

82 campos:
['OBJECTID', 'Shape', 'Join_Count', 'TARGET_FID', 'Join_Count_1', 'TARGET_FID_1', 'name', 'relacion_superficie', 'estado_terreno', 'etiqueta', 'clave21', 'ORIG_FID', 'OBJECTID_1', 'numero_predial_nacional', 'numero_predial_anterior', 'condicion_predio', 'destinacion_economica', 'tipo_predio', 'estado', 'matricula_inmobiliaria', 'estado_fmi', 'fecha_apertura_fmi', 'codigo_homologado', 'nupre', 'fecha_inscripcion_catastral', 'area_catastral_terreno', 'area_registral_m2', 'vigencia_actualizacion', 'nombre', 'tipo', 'aplica_efectos_registrales', 'tipo_vivienda', 'estrato', 'clave21_1', 'OBJECTID_12', 'NOMBRE_1', 'LOCALIDAD', 'AREA_HAS', 'Shape_Leng', 'COD_BARRIO', 'BARRIOS_OK', 'COD_LOC', 'OBSERV', 'CS', 'ENUME', 'Estrato_1', 'Estrato_2', 'Estrato_3', 'Estrato_4', 'Estrato_5', 'Estrato_6', 'Res_No_Est', 'Sin_Datos', 'Tot_Pred', 'Tot_Pre_Es', 'Tot_E1_E2', 'Tot_RestoE', 'Porc_E1', 'Porc_E2', 'Porc_E3', 'Porc_E4', 'Porc_E5', 'Porc_E6', 'Porc_E99', 'Porc_SD', 'Porc_TP', 'Porc_TPE', 

In [6]:
import arcpy, pandas as pd, os
arcpy.env.overwriteOutput = True
cols = ["COD_BARRIO","NOMBRE_1","LOCALIDAD","destinacion_economica","condicion_predio",
        "estrato","valor_metro","valor_comercial_metro"]
df = pd.DataFrame(list(arcpy.da.SearchCursor("Predio_analisis_2026", cols)), columns=cols)
print("Predios sin barrio:", df.COD_BARRIO.isna().sum())
df = df[df.COD_BARRIO.notna()].copy()
df["estrato_n"] = pd.to_numeric(df.estrato.astype(str).str.extract(r"(\d)")[0], errors="coerce")
df["hab"] = df.destinacion_economica.astype(str).str.contains("abitacional", case=False)
df["ph"] = df.condicion_predio.astype(str).str.upper().str.startswith("PH")
for c in ["valor_metro", "valor_comercial_metro"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
moda = lambda s: s.mode().iloc[0] if s.notna().any() else None
g = df.groupby("COD_BARRIO")
res = pd.DataFrame({
    "barrio": g.NOMBRE_1.first(), "localidad": g.LOCALIDAD.first(),
    "n_predios": g.size(),
    "pct_habitacional": (g.hab.mean()*100).round(1),
    "pct_ph": (g.ph.mean()*100).round(1),
    "estrato_moda": g.estrato_n.agg(moda),
    "estrato_prom": g.estrato_n.mean().round(2),
    "valor_m2_med": g.valor_metro.median(),
    "valor_com_m2_med": g.valor_comercial_metro.median(),
    "uso_predom": g.destinacion_economica.agg(moda)})
res.to_csv(os.path.join(arcpy.mp.ArcGISProject("CURRENT").homeFolder, "resumen_barrios_2026.csv"),
           encoding="utf-8-sig")
out = "Barrios_analisis_2026"
arcpy.management.CopyFeatures("Barrios", out)
nuevos = [("n_predios","LONG"),("pct_habitacional","DOUBLE"),("pct_ph","DOUBLE"),
          ("estrato_moda","SHORT"),("estrato_prom","DOUBLE"),("valor_m2_med","DOUBLE"),
          ("valor_com_m2_med","DOUBLE"),("uso_predom","TEXT")]
for n, t in nuevos:
    arcpy.management.AddField(out, n, t)
with arcpy.da.UpdateCursor(out, ["COD_BARRIO"] + [n for n, _ in nuevos]) as cur:
    for row in cur:
        if row[0] in res.index:
            r = res.loc[row[0]]
            v = [r.n_predios, r.pct_habitacional, r.pct_ph, r.estrato_moda, r.estrato_prom,
                 r.valor_m2_med, r.valor_com_m2_med, r.uso_predom]
            v = [None if pd.isna(x) else x for x in v]
            v[0] = int(v[0]); v[3] = None if v[3] is None else int(v[3])
            cur.updateRow([row[0]] + v)
print("Barrios con datos:", len(res), "de", arcpy.management.GetCount(out)[0])
print(res.sort_values("valor_m2_med", ascending=False).head(5)[["barrio","n_predios","estrato_moda","valor_m2_med"]])

Predios sin barrio: 5120
Barrios con datos: 74 de 204
                   barrio  n_predios  estrato_moda  valor_m2_med
COD_BARRIO                                                      
27                AMÉRICA       5045           6.0     1778180.0
20          SAN FRANCISCO       9604           5.0     1567800.0
41                MIRAMAR      10987           4.0     1346700.0
24            EL CASTILLO       7730           6.0     1279700.0
22          VILLA COUNTRY      12223           6.0     1199300.0


In [7]:
import arcpy, pandas as pd, os
arcpy.env.overwriteOutput = True
kb = lambda loc, bar: f"{loc}-{bar}"
claves_b = [kb(a, b) for a, b in arcpy.da.SearchCursor("Barrios", ["COD_LOC", "COD_BARRIO"])]
print("Barrios:", len(claves_b), "| claves únicas:", len(set(claves_b)))
cols = ["COD_LOC","COD_BARRIO","NOMBRE_1","LOCALIDAD","destinacion_economica","condicion_predio",
        "estrato","valor_metro","valor_comercial_metro"]
df = pd.DataFrame(list(arcpy.da.SearchCursor("Predio_analisis_2026", cols)), columns=cols)
print("Predios sin barrio:", df.COD_BARRIO.isna().sum())
df = df[df.COD_BARRIO.notna()].copy()
df["kb"] = [kb(a, b) for a, b in zip(df.COD_LOC, df.COD_BARRIO)]
df["estrato_n"] = pd.to_numeric(df.estrato.astype(str).str.extract(r"(\d)")[0], errors="coerce")
df["hab"] = df.destinacion_economica.astype(str).str.contains("abitacional", case=False)
df["ph"] = df.condicion_predio.astype(str).str.upper().str.startswith("PH")
for c in ["valor_metro", "valor_comercial_metro"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
moda = lambda s: s.mode().iloc[0] if s.notna().any() else None
g = df.groupby("kb")
res = pd.DataFrame({
    "barrio": g.NOMBRE_1.first(), "localidad": g.LOCALIDAD.first(), "n_predios": g.size(),
    "pct_habitacional": (g.hab.mean()*100).round(1), "pct_ph": (g.ph.mean()*100).round(1),
    "estrato_moda": g.estrato_n.agg(moda), "estrato_prom": g.estrato_n.mean().round(2),
    "valor_m2_med": g.valor_metro.median(), "valor_com_m2_med": g.valor_comercial_metro.median(),
    "uso_predom": g.destinacion_economica.agg(moda)})
res.to_csv(os.path.join(arcpy.mp.ArcGISProject("CURRENT").homeFolder, "resumen_barrios_2026.csv"),
           encoding="utf-8-sig")
out = "Barrios_analisis_2026"
arcpy.management.CopyFeatures("Barrios", out)
nuevos = [("n_predios","LONG"),("pct_habitacional","DOUBLE"),("pct_ph","DOUBLE"),
          ("estrato_moda","SHORT"),("estrato_prom","DOUBLE"),("valor_m2_med","DOUBLE"),
          ("valor_com_m2_med","DOUBLE"),("uso_predom","TEXT")]
for n, t in nuevos:
    arcpy.management.AddField(out, n, t)
with arcpy.da.UpdateCursor(out, ["COD_LOC", "COD_BARRIO"] + [n for n, _ in nuevos]) as cur:
    for row in cur:
        k = kb(row[0], row[1])
        if k in res.index:
            r = res.loc[k]
            v = [r.n_predios, r.pct_habitacional, r.pct_ph, r.estrato_moda, r.estrato_prom,
                 r.valor_m2_med, r.valor_com_m2_med, r.uso_predom]
            v = [None if pd.isna(x) else x for x in v]
            v[0] = int(v[0]); v[3] = None if v[3] is None else int(v[3])
            cur.updateRow(row[:2] + v)
print("Barrios con datos:", len(res), "de", len(claves_b))
print(res.sort_values("valor_m2_med", ascending=False).head(5)[["barrio","localidad","n_predios","estrato_moda","valor_m2_med"]])

Barrios: 204 | claves únicas: 194
Predios sin barrio: 5120
Barrios con datos: 194 de 204
                            barrio                          localidad  \
kb                                                                      
01-27               PALMAS DEL RIO                             RIOMAR   
01-22                   BUENAVISTA                             RIOMAR   
01-21       PASEO DE LA CASTELLANA                             RIOMAR   
02-48            INDUSTRIAL VIA 40           NORTE - CENTRO HISTORICO   
01 y 02-14         ALTOS DEL PRADO  RIOMAR y NORTE - CENTRO HISTORICO   

            n_predios  estrato_moda  valor_m2_med  
kb                                                 
01-27            3819           6.0     1778180.0  
01-22            2722           6.0     1778180.0  
01-21            4011           6.0     1649540.0  
02-48            1948           5.0     1608000.0  
01 y 02-14      14655           6.0     1608000.0  


In [8]:
import pandas as pd
b = pd.DataFrame(list(arcpy.da.SearchCursor("Barrios", ["COD_LOC","COD_BARRIO","NOMBRE"])),
                 columns=["loc","bar","nombre"])
b["kb"] = b["loc"].astype(str) + "-" + b["bar"].astype(str)
print(b[b.kb.duplicated(keep=False)].sort_values("kb").to_string())

    loc bar                         nombre     kb
141  01  01                     LAS FLORES  01-01
157  01  01                        TAJAMAR  01-01
163  01  01         CORREGIMIENTO LA PLAYA  01-01
140  01  02               INDUSTRIAL NORTE  01-02
186  01  02                   PUERTO MOCHO  01-02
139  01  03                          SIAPE  01-03
144  01  03              PLAN ZONAL RIOMAR  01-03
95   02  04                     BARLOVENTO  02-04
168  02  04            CORREDOR VIA AL MAR  02-04
93   02  05                     EL ROSARIO  02-05
165  02  05                  BOSQUE URBANO  02-05
102  02  06                          ABAJO  02-06
202  02  06         PLANES PARCIALES NORTE  02-06
66   03  02                    LOMA FRESCA  03-02
164  03  02        CORREGIMIENTO JUAN MINA  03-02
83   03  08                  NUEVA GRANADA  03-08
201  03  08  PLANES PARCIALES SUROCCIDENTE  03-08
86   03  26                    EL SILENCIO  03-26
199  03  26                    LA LIBERTAD  03-26


In [9]:
import arcpy, pandas as pd, os
arcpy.env.overwriteOutput = True
kb = lambda loc, bar, nom: f"{loc}-{bar}-{nom}"
claves_b = [kb(*r) for r in arcpy.da.SearchCursor("Barrios", ["COD_LOC", "COD_BARRIO", "NOMBRE"])]
print("Barrios:", len(claves_b), "| claves únicas:", len(set(claves_b)))
cols = ["COD_LOC","COD_BARRIO","NOMBRE_1","LOCALIDAD","destinacion_economica","condicion_predio",
        "estrato","valor_metro","valor_comercial_metro"]
df = pd.DataFrame(list(arcpy.da.SearchCursor("Predio_analisis_2026", cols)), columns=cols)
df = df[df.COD_BARRIO.notna()].copy()
df["kb"] = [kb(a, b, c) for a, b, c in zip(df.COD_LOC, df.COD_BARRIO, df.NOMBRE_1)]
df["estrato_n"] = pd.to_numeric(df.estrato.astype(str).str.extract(r"(\d)")[0], errors="coerce")
df["hab"] = df.destinacion_economica.astype(str).str.contains("abitacional", case=False)
df["ph"] = df.condicion_predio.astype(str).str.upper().str.startswith("PH")
for c in ["valor_metro", "valor_comercial_metro"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
moda = lambda s: s.mode().iloc[0] if s.notna().any() else None
g = df.groupby("kb")
res = pd.DataFrame({
    "barrio": g.NOMBRE_1.first(), "localidad": g.LOCALIDAD.first(), "n_predios": g.size(),
    "pct_habitacional": (g.hab.mean()*100).round(1), "pct_ph": (g.ph.mean()*100).round(1),
    "estrato_moda": g.estrato_n.agg(moda), "estrato_prom": g.estrato_n.mean().round(2),
    "valor_m2_med": g.valor_metro.median(), "valor_com_m2_med": g.valor_comercial_metro.median(),
    "uso_predom": g.destinacion_economica.agg(moda)})
res.to_csv(os.path.join(arcpy.mp.ArcGISProject("CURRENT").homeFolder, "resumen_barrios_2026.csv"),
           encoding="utf-8-sig")
out = "Barrios_analisis_2026"
arcpy.management.CopyFeatures("Barrios", out)
nuevos = [("n_predios","LONG"),("pct_habitacional","DOUBLE"),("pct_ph","DOUBLE"),
          ("estrato_moda","SHORT"),("estrato_prom","DOUBLE"),("valor_m2_med","DOUBLE"),
          ("valor_com_m2_med","DOUBLE"),("uso_predom","TEXT")]
for n, t in nuevos:
    arcpy.management.AddField(out, n, t)
with arcpy.da.UpdateCursor(out, ["COD_LOC", "COD_BARRIO", "NOMBRE"] + [n for n, _ in nuevos]) as cur:
    for row in cur:
        k = kb(row[0], row[1], row[2])
        if k in res.index:
            r = res.loc[k]
            v = [r.n_predios, r.pct_habitacional, r.pct_ph, r.estrato_moda, r.estrato_prom,
                 r.valor_m2_med, r.valor_com_m2_med, r.uso_predom]
            v = [None if pd.isna(x) else x for x in v]
            v[0] = int(v[0]); v[3] = None if v[3] is None else int(v[3])
            cur.updateRow(row[:3] + v)
print("Barrios con datos:", len(res), "de", len(claves_b))
print(res.sort_values("valor_m2_med", ascending=False).head(5)[["barrio","localidad","n_predios","estrato_moda","valor_m2_med"]])

Barrios: 204 | claves únicas: 204
Barrios con datos: 203 de 204
                                              barrio  \
kb                                                     
01-27-PALMAS DEL RIO                  PALMAS DEL RIO   
01-22-BUENAVISTA                          BUENAVISTA   
01-21-PASEO DE LA CASTELLANA  PASEO DE LA CASTELLANA   
02-48-INDUSTRIAL VIA 40            INDUSTRIAL VIA 40   
02-23-EL GOLF                                EL GOLF   

                                             localidad  n_predios  \
kb                                                                  
01-27-PALMAS DEL RIO                            RIOMAR       3819   
01-22-BUENAVISTA                                RIOMAR       2722   
01-21-PASEO DE LA CASTELLANA                    RIOMAR       4011   
02-48-INDUSTRIAL VIA 40       NORTE - CENTRO HISTORICO       1948   
02-23-EL GOLF                 NORTE - CENTRO HISTORICO       1788   

                              estrato_moda  valor_m2_med  


In [10]:
print([f.name for f in arcpy.ListFields("Barrios_analisis_2026")][-10:])

['Shape_Length', 'Shape_Area', 'n_predios', 'pct_habitacional', 'pct_ph', 'estrato_moda', 'estrato_prom', 'valor_m2_med', 'valor_com_m2_med', 'uso_predom']


In [11]:
import arcpy, os
aprx = arcpy.mp.ArcGISProject("CURRENT")
m = aprx.listMaps("Map")[0]
for l in m.listLayers("Barrios_analisis_2026"):
    m.removeLayer(l)
lyr = m.addDataFromPath(os.path.join(arcpy.env.workspace, "Barrios_analisis_2026"))
sym = lyr.symbology
sym.updateRenderer("GraduatedColorsRenderer")
sym.renderer.classificationField = "valor_m2_med"
sym.renderer.breakCount = 5
lyr.symbology = sym
print("Capa cargada y simbolizada por valor_m2_med")

Capa cargada y simbolizada por valor_m2_med


In [12]:
arcpy.management.CalculateField("Barrios_analisis_2026", "dens_ha",
    "!n_predios! / (!Shape_Area! / 10000) if !n_predios! else None", "PYTHON3", field_type="DOUBLE")
print("Listo: campo dens_ha creado")

Listo: campo dens_ha creado


In [13]:
import arcpy, pandas as pd, os
# 1. Valor del suelo por estrato (nivel predio)
cols = ["estrato", "valor_comercial_metro"]
p = pd.DataFrame(list(arcpy.da.SearchCursor("Predio_analisis_2026", cols)), columns=cols)
p["estrato_n"] = pd.to_numeric(p.estrato.astype(str).str.extract(r"(\d)")[0], errors="coerce")
p["valor"] = pd.to_numeric(p.valor_comercial_metro, errors="coerce")
p = p.dropna(subset=["estrato_n", "valor"])
p = p[p.estrato_n.between(1, 6)]
t = p.groupby("estrato_n").valor.agg(predios="size", mediana="median", promedio="mean").round(0)
t["% predios"] = (t.predios / t.predios.sum() * 100).round(1)
print("VALOR COMERCIAL DEL SUELO (COP/m²) POR ESTRATO — nivel predio")
print(t.to_string())
# 2. Correlación entre barrios
b = pd.DataFrame(list(arcpy.da.SearchCursor("Barrios_analisis_2026",
        ["estrato_prom", "valor_com_m2_med", "pct_habitacional", "dens_ha"])),
        columns=["estrato_prom", "valor", "pct_hab", "dens_ha"]).dropna()
print("\nCORRELACIÓN entre barrios (n =", len(b), ")")
print("Pearson  estrato vs valor:", round(b.estrato_prom.corr(b.valor), 3))
print("Spearman estrato vs valor:", round(b.estrato_prom.corr(b.valor, method="spearman"), 3))
print("Spearman %habitacional vs valor:", round(b.pct_hab.corr(b.valor, method="spearman"), 3))
print("Spearman densidad vs valor:", round(b.dens_ha.corr(b.valor, method="spearman"), 3))
# 3. Guardar la tabla para el informe
t.to_csv(os.path.join(arcpy.mp.ArcGISProject("CURRENT").homeFolder, "valor_por_estrato.csv"),
         encoding="utf-8-sig")
print("\nTabla guardada: valor_por_estrato.csv")

VALOR COMERCIAL DEL SUELO (COP/m²) POR ESTRATO — nivel predio
           predios    mediana   promedio  % predios
estrato_n                                          
1.0         125833   179000.0   247163.0       31.1
2.0          84479   322000.0   309176.0       20.8
3.0          78247   619000.0   692364.0       19.3
4.0          62890  1204000.0  1353796.0       15.5
5.0          25990  1910000.0  1789701.0        6.4
6.0          27793  2300000.0  2186461.0        6.9

CORRELACIÓN entre barrios (n = 193 )
Pearson  estrato vs valor: 0.868
Spearman estrato vs valor: 0.855
Spearman %habitacional vs valor: -0.199
Spearman densidad vs valor: 0.134

Tabla guardada: valor_por_estrato.csv


In [1]:
import requests, re
BASE = "https://services3.arcgis.com/oGYAc07w6wsvgUYr/arcgis/rest/services"
CLAVES = ["increment", "zona", "zhg", "aval", "valor", "catastr", "predio", "construid", "barrios_", "estrat", "huella"]
serv = requests.get(BASE + "?f=json", timeout=60).json()["services"]
cand = [s for s in serv if any(k in s["name"].lower() for k in CLAVES)]
print(f"{len(serv)} servicios | {len(cand)} candidatos\n")
for s in cand:
    url = f"{BASE}/{s['name']}/{s['type']}"
    try:
        j = requests.get(url + "?f=json", timeout=30).json()
        for c in j.get("layers", []) + j.get("tables", []):
            lj = requests.get(f"{url}/{c['id']}?f=json", timeout=30).json()
            n = requests.get(f"{url}/{c['id']}/query", params={"where": "1=1", "returnCountOnly": "true", "f": "json"}, timeout=30).json().get("count")
            campos = [f["name"] for f in lj.get("fields", [])]
            anos = sorted(set(re.findall(r"(?:19|20)\d\d", s["name"] + " " + c["name"] + " " + " ".join(campos))))
            print(f"{s['name']} / {c['id']} {c['name']} | {n} reg | años: {anos}")
            print("   campos:", ", ".join(campos[:25]))
    except Exception as e:
        print("FALLA", s["name"], e)

1353 servicios | 113 candidatos

ACTUALIZACION_ZONA_RESTRICCION / 0 ACTUALIZACION - P NUEVOS CONSTRUSEÑALES | 16 reg | años: []
   campos: OBJECTID, DIRECCION_SIN_NORMALIZAR, DIRECCION_NORMALIZADA, SCORE, X, Y, Punto_de_vigilancia_, Valor, Caracteristicas
Área_construida_2022_WFL1 / 0 Predios_totales_2022 | 476252 reg | años: ['2022']
   campos: FID, OBJECTID, CODIGO, Shape_Leng, OBJECTID_1, Referencia, Total_Prop, MANZANA, SECTOR, BARRIO, LOCALIDAD, direccion, destino, area_terre, area_const, avaluo, Rango, Shape__Area, Shape__Length
Área_construida_2022_WFL1 / 1 c2022ene01_sec08001_ExcelToTable1 | 406247 reg | años: ['2022']
   campos: OBJECTID, Referencia, Total_Prop, MANZANA, SECTOR, BARRIO, LOCALIDAD, direccion, destino, area_terreno, area_construida, avaluo, Rango
Área_construida_2022_WFL1 / 2 loca_Const_Totales_20102022_ | 7 reg | años: ['2010', '2022']
   campos: OBJECTID, Loc_vig, Localidad, Cant__De_Predios, Area_de_Terreno, Área_de_construcción, Avaluo_catastral, Vigencia
Ár

In [1]:
Layouts automáticos para los mapas del proyecto Barranquilla (ArcGIS Pro 3.x).
 
Cómo usarlo:
  1. Abre tu proyecto en ArcGIS Pro.
  2. Ve a Analysis > Python > Python Window (o abre un Notebook nuevo).
  3. Edita la sección CONFIGURACIÓN (carpeta de salida y nombres de tus mapas).
  4. Pega todo el código y ejecútalo.
 
Crea un layout por mapa (título, subtítulo, mapa, leyenda, norte, escala,
fuente y autoría) y exporta cada uno como PNG a 300 dpi con el nombre
mapa1.png ... mapa6.png, listos para la presentación.
"""
import arcpy
import os
import datetime
 
# ===================== CONFIGURACIÓN =====================
OUT_DIR = r"C:\Users\TU_USUARIO\Documents\mapas_presentacion"   # cambia esta ruta
AUTOR = "Laura Valeria Ortiz Díaz"
FUENTE = "Fuente: Catastro abierto Alcaldía de Barranquilla (LADM-COL) · Zonas ZHGU 2025"
FUENTE_TITULO = "Segoe UI"        # fuente para títulos
FUENTE_TEXTO = "Segoe UI"         # fuente para textos
ANCHO, ALTO = 10, 10              # pulgadas (cuadrado = encaja en las diapositivas)
 
# (nombre EXACTO del mapa en el panel Catalog > Maps, título, subtítulo, archivo)
MAPAS = [
    ("Valor del suelo",   "Valor del suelo en Barranquilla",      "COP por m² · ZHGU 2025",                     "mapa1"),
    ("Estrato",           "Estrato predominante por barrio",      "Estrato socioeconómico 1 a 6",               "mapa2"),
    ("Uso residencial",   "Uso residencial por barrio",           "% de predios residenciales",                 "mapa3"),
    ("Densidad predial",  "Densidad predial",                     "Predios por hectárea",                       "mapa4"),
    ("Bivariado",         "Valor del suelo × estrato",            "Clasificación bivariada 3 × 3",              "mapa5"),
    ("Residuos",          "Residuos del modelo",                  "Dónde el suelo vale más o menos de lo esperado", "mapa6"),
]
# =========================================================
 
aprx = arcpy.mp.ArcGISProject("CURRENT")
os.makedirs(OUT_DIR, exist_ok=True)
 
 
def estilo(tipo, preferidos):
    """Busca un estilo de ArcGIS 2D por nombre; si no existe usa el primero disponible."""
    items = aprx.listStyleItems("ArcGIS 2D", tipo)
    for nombre in preferidos:
        for it in items:
            if it.name == nombre:
                return it
    return items[0] if items else None
 
 
def texto(lyt, x, y, txt, size, estilo_fuente="Regular", fuente=FUENTE_TEXTO, nombre=None):
    return aprx.createTextElement(lyt, arcpy.Point(x, y), "POINT", txt, size,
                                  fuente, estilo_fuente, name=nombre)
 
 
def capa_principal(m):
    """Primera capa de polígonos visible (para hacer zoom a ella)."""
    for lyr in m.listLayers():
        if lyr.isFeatureLayer and lyr.visible:
            try:
                if arcpy.Describe(lyr).shapeType == "Polygon":
                    return lyr
            except Exception:
                pass
    return None
 
 
ST_NORTE = estilo("NORTH_ARROW", ["ArcGIS North 3", "ArcGIS North 1"])
ST_ESCALA = estilo("SCALE_BAR", ["Scale Line 1 Metric", "Alternating Scale Bar 1 Metric"])
ST_LEYENDA = estilo("LEGEND", ["Legend 1"])
fecha = datetime.date.today().strftime("%Y")
 
for nombre_mapa, titulo, subtitulo, archivo in MAPAS:
    mapas = aprx.listMaps(nombre_mapa)
    if not mapas:
        print(f"⚠ No encontré el mapa '{nombre_mapa}'. Revisa el nombre en Catalog > Maps.")
        continue
    m = mapas[0]
 
    # Borra un layout anterior con el mismo nombre para poder re-ejecutar el script
    nombre_lyt = f"Layout - {titulo}"
    for viejo in aprx.listLayouts(nombre_lyt):
        aprx.deleteItem(viejo)
 
    lyt = aprx.createLayout(ANCHO, ALTO, "INCH", nombre_lyt)
 
    # --- Encabezado ---
    texto(lyt, 0.5, 9.3, titulo, 28, "Bold", FUENTE_TITULO, "Titulo")
    texto(lyt, 0.5, 8.9, subtitulo, 15, "Regular", FUENTE_TEXTO, "Subtitulo")
 
    # --- Marco del mapa ---
    mf = lyt.createMapFrame(arcpy.Extent(0.5, 2.1, 9.5, 8.6), m, "Mapa")
    lyr = capa_principal(m)
    if lyr:
        mf.camera.setExtent(mf.getLayerExtent(lyr, False, True))
        mf.camera.scale = mf.camera.scale * 1.05   # un poco de aire alrededor
 
    # --- Leyenda (abajo a la izquierda) ---
    if ST_LEYENDA:
        ley = lyt.createMapSurroundElement(arcpy.Point(0.5, 0.6), "LEGEND", mf, ST_LEYENDA, "Leyenda")
        ley.elementPositionX, ley.elementPositionY = 0.5, 0.6
        ley.elementWidth, ley.elementHeight = 5.2, 1.35
        try:
            ley.fittingStrategy = "AdjustColumnsAndFont"
        except Exception:
            pass
 
    # --- Norte y escala (abajo a la derecha) ---
    if ST_NORTE:
        norte = lyt.createMapSurroundElement(arcpy.Point(9.0, 1.2), "NORTH_ARROW", mf, ST_NORTE, "Norte")
        norte.elementHeight = 0.7
        norte.elementPositionX, norte.elementPositionY = 8.9, 1.15
    if ST_ESCALA:
        esc = lyt.createMapSurroundElement(arcpy.Point(6.2, 1.2), "SCALE_BAR", mf, ST_ESCALA, "Escala")
        esc.elementWidth = 2.4
        esc.elementPositionX, esc.elementPositionY = 6.2, 1.2
 
    # --- Pie: fuente, sistema de referencia y autoría ---
    sr = m.spatialReference.name.replace("_", " ") if m.spatialReference else ""
    texto(lyt, 6.2, 0.85, f"Sistema de referencia: {sr}", 8, nombre="SRC")
    texto(lyt, 0.5, 0.3, FUENTE, 8, nombre="Fuente")
    texto(lyt, 6.2, 0.6, f"Elaboró: {AUTOR} · {fecha}", 8, "Bold", nombre="Autor")
 
    # --- Exportar ---
    salida = os.path.join(OUT_DIR, f"{archivo}.png")
    lyt.exportToPNG(salida, resolution=300)
    print(f"✓ {titulo} -> {salida}")
 
aprx.save()
print("Listo. Abre los layouts en Catalog > Layouts para ajustar detalles.")
 

<class 'SyntaxError'>: invalid decimal literal (C:/Users/sopor/AppData/Local/Temp/ArcGISProTemp11080/xpython_11080/3125499684.py, line 1)

In [2]:
import arcpy
import os
import datetime
 
# ===================== CONFIGURACIÓN =====================
OUT_DIR = r"C:\Users\TU_USUARIO\Documents\mapas_presentacion"   # cambia esta ruta
AUTOR = "Laura Valeria Ortiz Díaz"
FUENTE = "Fuente: Catastro abierto Alcaldía de Barranquilla (LADM-COL) · Zonas ZHGU 2025"
FUENTE_TITULO = "Segoe UI"        # fuente para títulos
FUENTE_TEXTO = "Segoe UI"         # fuente para textos
ANCHO, ALTO = 10, 10              # pulgadas (cuadrado = encaja en las diapositivas)
 
# (nombre EXACTO del mapa en el panel Catalog > Maps, título, subtítulo, archivo)
MAPAS = [
    ("Valor del suelo",   "Valor del suelo en Barranquilla",      "COP por m² · ZHGU 2025",                     "mapa1"),
    ("Estrato",           "Estrato predominante por barrio",      "Estrato socioeconómico 1 a 6",               "mapa2"),
    ("Uso residencial",   "Uso residencial por barrio",           "% de predios residenciales",                 "mapa3"),
    ("Densidad predial",  "Densidad predial",                     "Predios por hectárea",                       "mapa4"),
    ("Bivariado",         "Valor del suelo × estrato",            "Clasificación bivariada 3 × 3",              "mapa5"),
    ("Residuos",          "Residuos del modelo",                  "Dónde el suelo vale más o menos de lo esperado", "mapa6"),
]
# =========================================================
 
aprx = arcpy.mp.ArcGISProject("CURRENT")
os.makedirs(OUT_DIR, exist_ok=True)
 
 
def estilo(tipo, preferidos):
    """Busca un estilo de ArcGIS 2D por nombre; si no existe usa el primero disponible."""
    items = aprx.listStyleItems("ArcGIS 2D", tipo)
    for nombre in preferidos:
        for it in items:
            if it.name == nombre:
                return it
    return items[0] if items else None
 
 
def texto(lyt, x, y, txt, size, estilo_fuente="Regular", fuente=FUENTE_TEXTO, nombre=None):
    return aprx.createTextElement(lyt, arcpy.Point(x, y), "POINT", txt, size,
                                  fuente, estilo_fuente, name=nombre)
 
 
def capa_principal(m):
    """Primera capa de polígonos visible (para hacer zoom a ella)."""
    for lyr in m.listLayers():
        if lyr.isFeatureLayer and lyr.visible:
            try:
                if arcpy.Describe(lyr).shapeType == "Polygon":
                    return lyr
            except Exception:
                pass
    return None
 
 
ST_NORTE = estilo("NORTH_ARROW", ["ArcGIS North 3", "ArcGIS North 1"])
ST_ESCALA = estilo("SCALE_BAR", ["Scale Line 1 Metric", "Alternating Scale Bar 1 Metric"])
ST_LEYENDA = estilo("LEGEND", ["Legend 1"])
fecha = datetime.date.today().strftime("%Y")
 
for nombre_mapa, titulo, subtitulo, archivo in MAPAS:
    mapas = aprx.listMaps(nombre_mapa)
    if not mapas:
        print(f"⚠ No encontré el mapa '{nombre_mapa}'. Revisa el nombre en Catalog > Maps.")
        continue
    m = mapas[0]
 
    # Borra un layout anterior con el mismo nombre para poder re-ejecutar el script
    nombre_lyt = f"Layout - {titulo}"
    for viejo in aprx.listLayouts(nombre_lyt):
        aprx.deleteItem(viejo)
 
    lyt = aprx.createLayout(ANCHO, ALTO, "INCH", nombre_lyt)
 
    # --- Encabezado ---
    texto(lyt, 0.5, 9.3, titulo, 28, "Bold", FUENTE_TITULO, "Titulo")
    texto(lyt, 0.5, 8.9, subtitulo, 15, "Regular", FUENTE_TEXTO, "Subtitulo")
 
    # --- Marco del mapa ---
    mf = lyt.createMapFrame(arcpy.Extent(0.5, 2.1, 9.5, 8.6), m, "Mapa")
    lyr = capa_principal(m)
    if lyr:
        mf.camera.setExtent(mf.getLayerExtent(lyr, False, True))
        mf.camera.scale = mf.camera.scale * 1.05   # un poco de aire alrededor
 
    # --- Leyenda (abajo a la izquierda) ---
    if ST_LEYENDA:
        ley = lyt.createMapSurroundElement(arcpy.Point(0.5, 0.6), "LEGEND", mf, ST_LEYENDA, "Leyenda")
        ley.elementPositionX, ley.elementPositionY = 0.5, 0.6
        ley.elementWidth, ley.elementHeight = 5.2, 1.35
        try:
            ley.fittingStrategy = "AdjustColumnsAndFont"
        except Exception:
            pass
 
    # --- Norte y escala (abajo a la derecha) ---
    if ST_NORTE:
        norte = lyt.createMapSurroundElement(arcpy.Point(9.0, 1.2), "NORTH_ARROW", mf, ST_NORTE, "Norte")
        norte.elementHeight = 0.7
        norte.elementPositionX, norte.elementPositionY = 8.9, 1.15
    if ST_ESCALA:
        esc = lyt.createMapSurroundElement(arcpy.Point(6.2, 1.2), "SCALE_BAR", mf, ST_ESCALA, "Escala")
        esc.elementWidth = 2.4
        esc.elementPositionX, esc.elementPositionY = 6.2, 1.2
 
    # --- Pie: fuente, sistema de referencia y autoría ---
    sr = m.spatialReference.name.replace("_", " ") if m.spatialReference else ""
    texto(lyt, 6.2, 0.85, f"Sistema de referencia: {sr}", 8, nombre="SRC")
    texto(lyt, 0.5, 0.3, FUENTE, 8, nombre="Fuente")
    texto(lyt, 6.2, 0.6, f"Elaboró: {AUTOR} · {fecha}", 8, "Bold", nombre="Autor")
 
    # --- Exportar ---
    salida = os.path.join(OUT_DIR, f"{archivo}.png")
    lyt.exportToPNG(salida, resolution=300)
    print(f"✓ {titulo} -> {salida}")
 
aprx.save()
print("Listo. Abre los layouts en Catalog > Layouts para ajustar detalles.")

<class 'PermissionError'>: [WinError 5] Acceso denegado: 'C:\\Users\\TU_USUARIO'

In [3]:
import arcpy
import os
import datetime
 
# ===================== CONFIGURACIÓN =====================
OUT_DIR = ""   # déjalo vacío: guarda en la carpeta del proyecto, subcarpeta "mapas_presentacion"
AUTOR = "Laura Valeria Ortiz Díaz"
FUENTE = "Fuente: Catastro abierto Alcaldía de Barranquilla (LADM-COL) · Zonas ZHGU 2025"
FUENTE_TITULO = "Segoe UI"        # fuente para títulos
FUENTE_TEXTO = "Segoe UI"         # fuente para textos
ANCHO, ALTO = 10, 10              # pulgadas (cuadrado = encaja en las diapositivas)
 
# (nombre EXACTO del mapa en el panel Catalog > Maps, título, subtítulo, archivo)
MAPAS = [
    ("Valor del suelo",   "Valor del suelo en Barranquilla",      "COP por m² · ZHGU 2025",                     "mapa1"),
    ("Estrato",           "Estrato predominante por barrio",      "Estrato socioeconómico 1 a 6",               "mapa2"),
    ("Uso residencial",   "Uso residencial por barrio",           "% de predios residenciales",                 "mapa3"),
    ("Densidad predial",  "Densidad predial",                     "Predios por hectárea",                       "mapa4"),
    ("Bivariado",         "Valor del suelo × estrato",            "Clasificación bivariada 3 × 3",              "mapa5"),
    ("Residuos",          "Residuos del modelo",                  "Dónde el suelo vale más o menos de lo esperado", "mapa6"),
]
# =========================================================
 
aprx = arcpy.mp.ArcGISProject("CURRENT")
if not OUT_DIR:
    OUT_DIR = os.path.join(aprx.homeFolder, "mapas_presentacion")
print("Guardando imágenes en:", OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)
 
 
def estilo(tipo, preferidos):
    """Busca un estilo de ArcGIS 2D por nombre; si no existe usa el primero disponible."""
    items = aprx.listStyleItems("ArcGIS 2D", tipo)
    for nombre in preferidos:
        for it in items:
            if it.name == nombre:
                return it
    return items[0] if items else None
 
 
def texto(lyt, x, y, txt, size, estilo_fuente="Regular", fuente=FUENTE_TEXTO, nombre=None):
    return aprx.createTextElement(lyt, arcpy.Point(x, y), "POINT", txt, size,
                                  fuente, estilo_fuente, name=nombre)
 
 
def capa_principal(m):
    """Primera capa de polígonos visible (para hacer zoom a ella)."""
    for lyr in m.listLayers():
        if lyr.isFeatureLayer and lyr.visible:
            try:
                if arcpy.Describe(lyr).shapeType == "Polygon":
                    return lyr
            except Exception:
                pass
    return None
 
 
ST_NORTE = estilo("NORTH_ARROW", ["ArcGIS North 3", "ArcGIS North 1"])
ST_ESCALA = estilo("SCALE_BAR", ["Scale Line 1 Metric", "Alternating Scale Bar 1 Metric"])
ST_LEYENDA = estilo("LEGEND", ["Legend 1"])
fecha = datetime.date.today().strftime("%Y")
 
for nombre_mapa, titulo, subtitulo, archivo in MAPAS:
    mapas = aprx.listMaps(nombre_mapa)
    if not mapas:
        print(f"⚠ No encontré el mapa '{nombre_mapa}'. Revisa el nombre en Catalog > Maps.")
        continue
    m = mapas[0]
 
    # Borra un layout anterior con el mismo nombre para poder re-ejecutar el script
    nombre_lyt = f"Layout - {titulo}"
    for viejo in aprx.listLayouts(nombre_lyt):
        aprx.deleteItem(viejo)
 
    lyt = aprx.createLayout(ANCHO, ALTO, "INCH", nombre_lyt)
 
    # --- Encabezado ---
    texto(lyt, 0.5, 9.3, titulo, 28, "Bold", FUENTE_TITULO, "Titulo")
    texto(lyt, 0.5, 8.9, subtitulo, 15, "Regular", FUENTE_TEXTO, "Subtitulo")
 
    # --- Marco del mapa ---
    mf = lyt.createMapFrame(arcpy.Extent(0.5, 2.1, 9.5, 8.6), m, "Mapa")
    lyr = capa_principal(m)
    if lyr:
        mf.camera.setExtent(mf.getLayerExtent(lyr, False, True))
        mf.camera.scale = mf.camera.scale * 1.05   # un poco de aire alrededor
 
    # --- Leyenda (abajo a la izquierda) ---
    if ST_LEYENDA:
        ley = lyt.createMapSurroundElement(arcpy.Point(0.5, 0.6), "LEGEND", mf, ST_LEYENDA, "Leyenda")
        ley.elementPositionX, ley.elementPositionY = 0.5, 0.6
        ley.elementWidth, ley.elementHeight = 5.2, 1.35
        try:
            ley.fittingStrategy = "AdjustColumnsAndFont"
        except Exception:
            pass
 
    # --- Norte y escala (abajo a la derecha) ---
    if ST_NORTE:
        norte = lyt.createMapSurroundElement(arcpy.Point(9.0, 1.2), "NORTH_ARROW", mf, ST_NORTE, "Norte")
        norte.elementHeight = 0.7
        norte.elementPositionX, norte.elementPositionY = 8.9, 1.15
    if ST_ESCALA:
        esc = lyt.createMapSurroundElement(arcpy.Point(6.2, 1.2), "SCALE_BAR", mf, ST_ESCALA, "Escala")
        esc.elementWidth = 2.4
        esc.elementPositionX, esc.elementPositionY = 6.2, 1.2
 
    # --- Pie: fuente, sistema de referencia y autoría ---
    sr = m.spatialReference.name.replace("_", " ") if m.spatialReference else ""
    texto(lyt, 6.2, 0.85, f"Sistema de referencia: {sr}", 8, nombre="SRC")
    texto(lyt, 0.5, 0.3, FUENTE, 8, nombre="Fuente")
    texto(lyt, 6.2, 0.6, f"Elaboró: {AUTOR} · {fecha}", 8, "Bold", nombre="Autor")
 
    # --- Exportar ---
    salida = os.path.join(OUT_DIR, f"{archivo}.png")
    lyt.exportToPNG(salida, resolution=300)
    print(f"✓ {titulo} -> {salida}")
 
aprx.save()
print("Listo. Abre los layouts en Catalog > Layouts para ajustar detalles.")

Guardando imágenes en: D:\ProyectoCroma\Proyecto Barranquilla\Catastro Barranquilla\mapas_presentacion
⚠ No encontré el mapa 'Valor del suelo'. Revisa el nombre en Catalog > Maps.
⚠ No encontré el mapa 'Estrato'. Revisa el nombre en Catalog > Maps.
⚠ No encontré el mapa 'Uso residencial'. Revisa el nombre en Catalog > Maps.
⚠ No encontré el mapa 'Densidad predial'. Revisa el nombre en Catalog > Maps.
⚠ No encontré el mapa 'Bivariado'. Revisa el nombre en Catalog > Maps.
⚠ No encontré el mapa 'Residuos'. Revisa el nombre en Catalog > Maps.
Listo. Abre los layouts en Catalog > Layouts para ajustar detalles.


In [4]:
import arcpy
import os
import datetime
 
# ===================== CONFIGURACIÓN =====================
OUT_DIR = ""   # vacío = carpeta del proyecto, subcarpeta "mapas_presentacion"
MAPA_ORIGEN = "Map"               # nombre del mapa que contiene las capas
AUTOR = "Laura Valeria Ortiz Díaz"
FUENTE = "Fuente: Catastro abierto Alcaldía de Barranquilla (LADM-COL) · Zonas ZHGU 2025"
FUENTE_TITULO = "Segoe UI"
FUENTE_TEXTO = "Segoe UI"
ANCHO, ALTO = 10, 10              # pulgadas (cuadrado = encaja en las diapositivas)
 
# Capas que se dejan encendidas en TODOS los mapas (mapa base)
CAPAS_BASE = ["World Topographic Map", "World_Hillshade"]
 
# (nombre de la CAPA en Drawing Order, título, subtítulo, archivo)
TEMAS = [
    ("Neighborhoods – Land value 2025",          "Valor del suelo en Barranquilla",  "COP por m² · ZHGU 2025",                          "mapa1"),
    ("Predominant socioeconomic stratum",        "Estrato predominante por barrio",  "Estrato socioeconómico 1 a 6",                    "mapa2"),
    ("Residential land use (%)",                 "Uso residencial por barrio",       "% de predios residenciales",                      "mapa3"),
    ("Property density (properties/ha)",         "Densidad predial",                 "Predios por hectárea",                            "mapa4"),
    ("Land value and stratum (bivariate)",       "Valor del suelo × estrato",        "Clasificación bivariada 3 × 3",                   "mapa5"),
    ("Land value vs. stratum (model residuals)", "Residuos del modelo",              "Dónde el suelo vale más o menos de lo esperado",  "mapa6"),
]
# =========================================================
 
 
def norm(s):
    """Compara nombres sin importar mayúsculas, espacios ni tipo de guion."""
    return " ".join(s.replace("–", "-").replace("—", "-").lower().split())
 
 
aprx = arcpy.mp.ArcGISProject("CURRENT")
if not OUT_DIR:
    OUT_DIR = os.path.join(aprx.homeFolder, "mapas_presentacion")
print("Guardando imágenes en:", OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)
 
origen = aprx.listMaps(MAPA_ORIGEN)
if not origen:
    raise SystemExit(f"No encontré el mapa '{MAPA_ORIGEN}'. Revisa el nombre en Catalog > Maps.")
origen = origen[0]
 
 
def estilo(tipo, preferidos):
    items = aprx.listStyleItems("ArcGIS 2D", tipo)
    for nombre in preferidos:
        for it in items:
            if it.name == nombre:
                return it
    return items[0] if items else None
 
 
def texto(lyt, x, y, txt, size, estilo_fuente="Regular", fuente=FUENTE_TEXTO, nombre=None):
    return aprx.createTextElement(lyt, arcpy.Point(x, y), "POINT", txt, size,
                                  fuente, estilo_fuente, name=nombre)
 
 
def capas_superiores(m):
    """Capas del primer nivel (no las subcapas de un grupo)."""
    return [l for l in m.listLayers() if l.longName == l.name]
 
 
def buscar(m, nombre):
    for l in capas_superiores(m):
        if norm(l.name) == norm(nombre):
            return l
    return None
 
 
def preparar_mapa(nombre_copia, capa_tema):
    """Copia el mapa original y deja visible solo el tema y el mapa base."""
    for viejo in aprx.listMaps(nombre_copia):
        aprx.deleteItem(viejo)
    m = aprx.copyItem(origen, nombre_copia)
    base = {norm(b) for b in CAPAS_BASE}
    for l in capas_superiores(m):
        l.visible = norm(l.name) in base or norm(l.name) == norm(capa_tema)
    return m
 
 
ST_NORTE = estilo("NORTH_ARROW", ["ArcGIS North 3", "ArcGIS North 1"])
ST_ESCALA = estilo("SCALE_BAR", ["Scale Line 1 Metric", "Alternating Scale Bar 1 Metric"])
ST_LEYENDA = estilo("LEGEND", ["Legend 1"])
anio = datetime.date.today().strftime("%Y")
 
for capa_tema, titulo, subtitulo, archivo in TEMAS:
    if not buscar(origen, capa_tema):
        print(f"⚠ No encontré la capa '{capa_tema}' en '{MAPA_ORIGEN}'. La salto.")
        continue
 
    m = preparar_mapa(f"Mapa - {titulo}", capa_tema)
    tema = buscar(m, capa_tema)
 
    nombre_lyt = f"Layout - {titulo}"
    for viejo in aprx.listLayouts(nombre_lyt):
        aprx.deleteItem(viejo)
    lyt = aprx.createLayout(ANCHO, ALTO, "INCH", nombre_lyt)
 
    # --- Encabezado ---
    texto(lyt, 0.5, 9.3, titulo, 28, "Bold", FUENTE_TITULO, "Titulo")
    texto(lyt, 0.5, 8.9, subtitulo, 15, "Regular", FUENTE_TEXTO, "Subtitulo")
 
    # --- Marco del mapa, con zoom a la capa del tema ---
    mf = lyt.createMapFrame(arcpy.Extent(0.5, 2.1, 9.5, 8.6), m, "Mapa")
    try:
        mf.camera.setExtent(mf.getLayerExtent(tema, False, True))
        mf.camera.scale = mf.camera.scale * 1.05
    except Exception as e:
        print(f"  (no pude hacer zoom a la capa: {e})")
 
    # --- Leyenda: solo la capa del tema ---
    if ST_LEYENDA:
        ley = lyt.createMapSurroundElement(arcpy.Point(0.5, 0.6), "LEGEND", mf, ST_LEYENDA, "Leyenda")
        try:
            for item in list(ley.items):
                if norm(item.name) != norm(tema.name):
                    ley.removeItem(item)
        except Exception:
            pass
        ley.elementPositionX, ley.elementPositionY = 0.5, 0.6
        ley.elementWidth, ley.elementHeight = 5.2, 1.35
        try:
            ley.fittingStrategy = "AdjustColumnsAndFont"
        except Exception:
            pass
 
    # --- Norte y escala ---
    if ST_NORTE:
        norte = lyt.createMapSurroundElement(arcpy.Point(9.0, 1.2), "NORTH_ARROW", mf, ST_NORTE, "Norte")
        norte.elementHeight = 0.7
        norte.elementPositionX, norte.elementPositionY = 8.9, 1.15
    if ST_ESCALA:
        esc = lyt.createMapSurroundElement(arcpy.Point(6.2, 1.2), "SCALE_BAR", mf, ST_ESCALA, "Escala")
        esc.elementWidth = 2.4
        esc.elementPositionX, esc.elementPositionY = 6.2, 1.2
 
    # --- Pie ---
    sr = m.spatialReference.name.replace("_", " ") if m.spatialReference else ""
    texto(lyt, 6.2, 0.85, f"Sistema de referencia: {sr}", 8, nombre="SRC")
    texto(lyt, 6.2, 0.6, f"Elaboró: {AUTOR} · {anio}", 8, "Bold", nombre="Autor")
    texto(lyt, 0.5, 0.3, FUENTE, 8, nombre="Fuente")
 
    salida = os.path.join(OUT_DIR, f"{archivo}.png")
    lyt.exportToPNG(salida, resolution=300)
    print(f"✓ {titulo} -> {salida}")
 
aprx.save()
print("Listo. Los layouts quedan en Catalog > Layouts para ajustar detalles a mano.")

Guardando imágenes en: D:\ProyectoCroma\Proyecto Barranquilla\Catastro Barranquilla\mapas_presentacion
✓ Valor del suelo en Barranquilla -> D:\ProyectoCroma\Proyecto Barranquilla\Catastro Barranquilla\mapas_presentacion\mapa1.png
✓ Estrato predominante por barrio -> D:\ProyectoCroma\Proyecto Barranquilla\Catastro Barranquilla\mapas_presentacion\mapa2.png
✓ Uso residencial por barrio -> D:\ProyectoCroma\Proyecto Barranquilla\Catastro Barranquilla\mapas_presentacion\mapa3.png
✓ Densidad predial -> D:\ProyectoCroma\Proyecto Barranquilla\Catastro Barranquilla\mapas_presentacion\mapa4.png
✓ Valor del suelo × estrato -> D:\ProyectoCroma\Proyecto Barranquilla\Catastro Barranquilla\mapas_presentacion\mapa5.png
✓ Residuos del modelo -> D:\ProyectoCroma\Proyecto Barranquilla\Catastro Barranquilla\mapas_presentacion\mapa6.png
Listo. Los layouts quedan en Catalog > Layouts para ajustar detalles a mano.


In [5]:
import arcpy
import os
import datetime
 
# ===================== CONFIGURACIÓN =====================
OUT_DIR = ""                      # vacío = carpeta del proyecto, subcarpeta "mapas_presentacion"
MAPA_ORIGEN = "Map"               # nombre del mapa que contiene las capas
AUTOR = "Laura Valeria Ortiz Díaz"
FUENTES = ("Fuente: Catastro abierto Alcaldía de Barranquilla (LADM-COL) · Zonas ZHGU 2025\n"
           "Mapa base: Esri, TomTom, Garmin, FAO, NOAA, USGS, © OpenStreetMap contributors")
FUENTE_LETRA = "Segoe UI"
ANCHO, ALTO = 13.333, 7.5         # pulgadas, 16:9 (igual que una diapositiva)
 
# Mapa base para los layouts. "Oceans" se parece a tu referencia (mar azul claro,
# tierra clara con relieve). Pon "" para dejar tu mapa base actual.
MAPA_BASE = "Oceans"
CAPAS_BASE = ["World Topographic Map", "World_Hillshade"]   # tu mapa base actual
 
COLOR_TITULO = (255, 255, 255)    # blanco
COLOR_HALO = (70, 110, 140)       # contorno azul grisáceo para que se lea sobre el mar
COLOR_FUENTES = (110, 110, 110)   # gris
 
MOSTRAR_ESCALA = True
MOSTRAR_NORTE = True
 
# (nombre de la CAPA en Drawing Order, título (2 líneas), subtítulo, archivo)
TEMAS = [
    ("Neighborhoods – Land value 2025",          "VALOR DEL SUELO\nBARRANQUILLA",     "COP por m² · ZHGU 2025",                          "mapa1"),
    ("Predominant socioeconomic stratum",        "ESTRATO\nPREDOMINANTE",             "Estrato socioeconómico por barrio",               "mapa2"),
    ("Residential land use (%)",                 "USO\nRESIDENCIAL",                  "% de predios residenciales por barrio",           "mapa3"),
    ("Property density (properties/ha)",         "DENSIDAD\nPREDIAL",                 "Predios por hectárea",                            "mapa4"),
    ("Land value and stratum (bivariate)",       "VALOR DEL SUELO\n× ESTRATO",        "Clasificación bivariada 3 × 3",                   "mapa5"),
    ("Land value vs. stratum (model residuals)", "RESIDUOS\nDEL MODELO",              "Dónde el suelo vale más o menos de lo esperado",  "mapa6"),
]
# =========================================================
 
 
def norm(s):
    return " ".join(s.replace("–", "-").replace("—", "-").lower().split())
 
 
aprx = arcpy.mp.ArcGISProject("CURRENT")
if not OUT_DIR:
    OUT_DIR = os.path.join(aprx.homeFolder, "mapas_presentacion")
print("Guardando imágenes en:", OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)
 
origen = aprx.listMaps(MAPA_ORIGEN)
if not origen:
    raise SystemExit(f"No encontré el mapa '{MAPA_ORIGEN}'. Revisa el nombre en Catalog > Maps.")
origen = origen[0]
 
 
# ---------- utilidades ----------
def estilo(tipo, preferidos):
    items = aprx.listStyleItems("ArcGIS 2D", tipo)
    for nombre in preferidos:
        for it in items:
            if it.name == nombre:
                return it
    return items[0] if items else None
 
 
def rgb(c):
    col = arcpy.cim.CreateCIMObjectFromClassName("CIMRGBColor", "V3")
    col.values = [c[0], c[1], c[2], 100]
    return col
 
 
def relleno(c):
    poly = arcpy.cim.CreateCIMObjectFromClassName("CIMPolygonSymbol", "V3")
    fill = arcpy.cim.CreateCIMObjectFromClassName("CIMSolidFill", "V3")
    fill.enable = True
    fill.color = rgb(c)
    poly.symbolLayers = [fill]
    return poly
 
 
def colorear(elm, color, halo=None, halo_tam=1.2, alineacion=None):
    """Cambia el color del texto, agrega halo y alineación (vía CIM)."""
    try:
        cim = elm.getDefinition("V3")
        ts = cim.graphic.symbol.symbol
        ts.symbol = relleno(color)
        if halo:
            ts.haloSymbol = relleno(halo)
            ts.haloSize = halo_tam
        if alineacion:
            ts.horizontalAlignment = alineacion
        elm.setDefinition(cim)
    except Exception as e:
        print(f"  (no pude cambiar el color de un texto: {e})")
 
 
def colocar(elm, x, y, ancla="TopLeftCorner", w=None, h=None):
    """Fija el punto de anclaje del elemento y lo ubica en (x, y)."""
    try:
        cim = elm.getDefinition("V3")
        cim.anchor = ancla
        elm.setDefinition(cim)
    except Exception:
        pass
    if w:
        elm.elementWidth = w
    if h:
        elm.elementHeight = h
    elm.elementPositionX, elm.elementPositionY = x, y
 
 
def sin_marco(elm):
    """Quita borde y fondo a un marco de mapa o leyenda."""
    try:
        cim = elm.getDefinition("V3")
        if getattr(cim, "graphicFrame", None):
            cim.graphicFrame.borderSymbol = None
            cim.graphicFrame.backgroundSymbol = None
        if hasattr(cim, "showTitle"):
            cim.showTitle = False
        elm.setDefinition(cim)
    except Exception as e:
        print(f"  (no pude quitar el marco: {e})")
 
 
def texto(lyt, txt, size, estilo_fuente="Regular", nombre=None):
    return aprx.createTextElement(lyt, arcpy.Point(1, 1), "POINT", txt, size,
                                  FUENTE_LETRA, estilo_fuente, name=nombre)
 
 
def capas_superiores(m):
    return [l for l in m.listLayers() if l.longName == l.name]
 
 
def buscar(m, nombre):
    for l in capas_superiores(m):
        if norm(l.name) == norm(nombre):
            return l
    return None
 
 
def preparar_mapa(nombre_copia, capa_tema):
    for viejo in aprx.listMaps(nombre_copia):
        aprx.deleteItem(viejo)
    m = aprx.copyItem(origen, nombre_copia)
    base = {norm(b) for b in CAPAS_BASE}
    for l in capas_superiores(m):
        if MAPA_BASE and norm(l.name) in base:
            m.removeLayer(l)            # se reemplaza por el mapa base nuevo
        else:
            l.visible = norm(l.name) in base or norm(l.name) == norm(capa_tema)
    if MAPA_BASE:
        try:
            m.addBasemap(MAPA_BASE)
        except Exception as e:
            print(f"  (no pude poner el mapa base '{MAPA_BASE}': {e})")
    return m
 
 
def encuadre(mf, capa, ancho, alto):
    """Deja la ciudad a la derecha del centro, para que el título y la
    leyenda queden sobre el mar y la tierra vacía de la izquierda."""
    e = mf.getLayerExtent(capa, False, True)
    ew, eh = e.XMax - e.XMin, e.YMax - e.YMin
    aspecto = ancho / alto
    h = eh * 1.12
    w = h * aspecto
    if w < ew * 1.6:
        w = ew * 1.6
        h = w / aspecto
    cx, cy = (e.XMin + e.XMax) / 2, (e.YMin + e.YMax) / 2
    xmin = cx - 0.62 * w
    ymin = cy - 0.5 * h
    mf.camera.setExtent(arcpy.Extent(xmin, ymin, xmin + w, ymin + h,
                                     spatial_reference=e.spatialReference))
 
 
ST_NORTE = estilo("NORTH_ARROW", ["ArcGIS North 3", "ArcGIS North 1"])
ST_ESCALA = estilo("SCALE_BAR", ["Scale Line 1 Metric", "Alternating Scale Bar 1 Metric"])
ST_LEYENDA = estilo("LEGEND", ["Legend 1"])
anio = datetime.date.today().strftime("%Y")
M = 0.15                     # margen blanco alrededor del mapa (pulgadas)
W, H = ANCHO, ALTO
 
for capa_tema, titulo, subtitulo, archivo in TEMAS:
    if not buscar(origen, capa_tema):
        print(f"⚠ No encontré la capa '{capa_tema}' en '{MAPA_ORIGEN}'. La salto.")
        continue
 
    nombre_titulo = titulo.replace("\n", " ").title()
    m = preparar_mapa(f"Mapa - {nombre_titulo}", capa_tema)
    tema = buscar(m, capa_tema)
 
    nombre_lyt = f"Layout - {nombre_titulo}"
    for viejo in aprx.listLayouts(nombre_lyt):
        aprx.deleteItem(viejo)
    lyt = aprx.createLayout(W, H, "INCH", nombre_lyt)
 
    # --- Mapa a toda página ---
    mf = lyt.createMapFrame(arcpy.Extent(M, M, W - M, H - M), m, "Mapa")
    sin_marco(mf)
    try:
        encuadre(mf, tema, W - 2 * M, H - 2 * M)
    except Exception as e:
        print(f"  (no pude encuadrar, uso zoom simple: {e})")
        mf.camera.setExtent(mf.getLayerExtent(tema, False, True))
 
    # --- Título grande arriba a la izquierda, sobre el mar ---
    t = texto(lyt, titulo, 44, "Bold", "Titulo")
    colorear(t, COLOR_TITULO, COLOR_HALO, 1.5)
    colocar(t, 0.55, H - 0.45)
 
    s = texto(lyt, subtitulo, 16, "Semibold", "Subtitulo")
    colorear(s, COLOR_TITULO, COLOR_HALO, 1.0)
    colocar(s, 0.6, H - 0.45 - 1.65)
 
    # --- Leyenda flotante abajo a la izquierda, sin marco ni título ---
    if ST_LEYENDA:
        ley = lyt.createMapSurroundElement(arcpy.Point(1, 1), "LEGEND", mf, ST_LEYENDA, "Leyenda")
        try:
            for item in list(ley.items):
                if norm(item.name) != norm(tema.name):
                    ley.removeItem(item)
        except Exception:
            pass
        sin_marco(ley)
        try:
            ley.fittingStrategy = "AdjustFontSize"
        except Exception:
            pass
        colocar(ley, 0.55, 0.5, ancla="BottomLeftCorner", w=3.4, h=2.6)
 
    # --- Norte arriba a la derecha, escala y fuentes abajo a la derecha ---
    if MOSTRAR_NORTE and ST_NORTE:
        n = lyt.createMapSurroundElement(arcpy.Point(1, 1), "NORTH_ARROW", mf, ST_NORTE, "Norte")
        colocar(n, W - 0.55, H - 0.45, ancla="TopRightCorner", h=0.55)
    if MOSTRAR_ESCALA and ST_ESCALA:
        esc = lyt.createMapSurroundElement(arcpy.Point(1, 1), "SCALE_BAR", mf, ST_ESCALA, "Escala")
        colocar(esc, W - 0.55, 1.05, ancla="BottomRightCorner", w=2.0)
 
    sr = m.spatialReference.name.replace("_", " ") if m.spatialReference else ""
    f = texto(lyt, f"{FUENTES}\nSistema de referencia: {sr} · Elaboró: {AUTOR}, {anio}", 8, nombre="Fuentes")
    colorear(f, COLOR_FUENTES, alineacion="Right")
    colocar(f, W - 0.55, 0.4, ancla="BottomRightCorner")
 
    salida = os.path.join(OUT_DIR, f"{archivo}.png")
    lyt.exportToPNG(salida, resolution=300)
    print(f"✓ {nombre_titulo} -> {salida}")
 
aprx.save()
print("Listo. Los layouts quedan en Catalog > Layouts por si quieres mover algo a mano.")

Guardando imágenes en: D:\ProyectoCroma\Proyecto Barranquilla\Catastro Barranquilla\mapas_presentacion
✓ Valor Del Suelo Barranquilla -> D:\ProyectoCroma\Proyecto Barranquilla\Catastro Barranquilla\mapas_presentacion\mapa1.png
✓ Estrato Predominante -> D:\ProyectoCroma\Proyecto Barranquilla\Catastro Barranquilla\mapas_presentacion\mapa2.png
✓ Uso Residencial -> D:\ProyectoCroma\Proyecto Barranquilla\Catastro Barranquilla\mapas_presentacion\mapa3.png
✓ Densidad Predial -> D:\ProyectoCroma\Proyecto Barranquilla\Catastro Barranquilla\mapas_presentacion\mapa4.png
✓ Valor Del Suelo × Estrato -> D:\ProyectoCroma\Proyecto Barranquilla\Catastro Barranquilla\mapas_presentacion\mapa5.png
✓ Residuos Del Modelo -> D:\ProyectoCroma\Proyecto Barranquilla\Catastro Barranquilla\mapas_presentacion\mapa6.png
Listo. Los layouts quedan en Catalog > Layouts por si quieres mover algo a mano.


In [6]:
import arcpy
import os
import datetime
 
# ===================== CONFIGURACIÓN =====================
OUT_DIR = ""                      # vacío = carpeta del proyecto, subcarpeta "mapas_presentacion"
MAPA_ORIGEN = "Map"               # nombre del mapa que contiene las capas
AUTOR = "Laura Valeria Ortiz Díaz"
FUENTES = ("Source: Barranquilla Mayor's Office open cadastre (LADM-COL) · ZHGU 2025 zones\n"
           "Basemap: Esri, TomTom, Garmin, FAO, NOAA, USGS, © OpenStreetMap contributors")
FUENTE_LETRA = "Segoe UI"
ANCHO, ALTO = 13.333, 7.5         # pulgadas, 16:9 (igual que una diapositiva)
 
# Mapa base para los layouts. "Oceans" se parece a tu referencia (mar azul claro,
# tierra clara con relieve). Pon "" para dejar tu mapa base actual.
MAPA_BASE = "Oceans"
CAPAS_BASE = ["World Topographic Map", "World_Hillshade"]   # tu mapa base actual
 
COLOR_TITULO = (255, 255, 255)    # blanco
COLOR_HALO = (70, 110, 140)       # contorno azul grisáceo para que se lea sobre el mar
COLOR_FUENTES = (110, 110, 110)   # gris
 
MOSTRAR_ESCALA = True
MOSTRAR_NORTE = True
 
# (nombre de la CAPA en Drawing Order, título (2 líneas), subtítulo, archivo)
TEMAS = [
    ("Neighborhoods – Land value 2025",          "LAND VALUE\nBARRANQUILLA",         "COP per m² · ZHGU 2025",                          "mapa1"),
    ("Predominant socioeconomic stratum",        "SOCIOECONOMIC\nSTRATUM",            "Predominant stratum by neighborhood",               "mapa2"),
    ("Residential land use (%)",                 "RESIDENTIAL\nLAND USE",             "% of residential properties by neighborhood",           "mapa3"),
    ("Property density (properties/ha)",         "PROPERTY\nDENSITY",                  "Properties per hectare",                            "mapa4"),
    ("Land value and stratum (bivariate)",       "LAND VALUE\n× STRATUM",             "Bivariate classification 3 × 3",                   "mapa5"),
    ("Land value vs. stratum (model residuals)", "MODEL\nRESIDUALS",                  "Where land is worth more or less than expected",  "mapa6"),
]
# =========================================================
 
 
def norm(s):
    return " ".join(s.replace("–", "-").replace("—", "-").lower().split())
 
 
aprx = arcpy.mp.ArcGISProject("CURRENT")
if not OUT_DIR:
    OUT_DIR = os.path.join(aprx.homeFolder, "mapas_presentacion")
print("Guardando imágenes en:", OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)
 
origen = aprx.listMaps(MAPA_ORIGEN)
if not origen:
    raise SystemExit(f"No encontré el mapa '{MAPA_ORIGEN}'. Revisa el nombre en Catalog > Maps.")
origen = origen[0]
 
 
# ---------- utilidades ----------
def estilo(tipo, preferidos):
    items = aprx.listStyleItems("ArcGIS 2D", tipo)
    for nombre in preferidos:
        for it in items:
            if it.name == nombre:
                return it
    return items[0] if items else None
 
 
def rgb(c):
    col = arcpy.cim.CreateCIMObjectFromClassName("CIMRGBColor", "V3")
    col.values = [c[0], c[1], c[2], 100]
    return col
 
 
def relleno(c):
    poly = arcpy.cim.CreateCIMObjectFromClassName("CIMPolygonSymbol", "V3")
    fill = arcpy.cim.CreateCIMObjectFromClassName("CIMSolidFill", "V3")
    fill.enable = True
    fill.color = rgb(c)
    poly.symbolLayers = [fill]
    return poly
 
 
def colorear(elm, color, halo=None, halo_tam=1.2, alineacion=None):
    """Cambia el color del texto, agrega halo y alineación (vía CIM)."""
    try:
        cim = elm.getDefinition("V3")
        ts = cim.graphic.symbol.symbol
        ts.symbol = relleno(color)
        if halo:
            ts.haloSymbol = relleno(halo)
            ts.haloSize = halo_tam
        if alineacion:
            ts.horizontalAlignment = alineacion
        elm.setDefinition(cim)
    except Exception as e:
        print(f"  (no pude cambiar el color de un texto: {e})")
 
 
def colocar(elm, x, y, ancla="TopLeftCorner", w=None, h=None):
    """Fija el punto de anclaje del elemento y lo ubica en (x, y)."""
    try:
        cim = elm.getDefinition("V3")
        cim.anchor = ancla
        elm.setDefinition(cim)
    except Exception:
        pass
    if w:
        elm.elementWidth = w
    if h:
        elm.elementHeight = h
    elm.elementPositionX, elm.elementPositionY = x, y
 
 
def sin_marco(elm):
    """Quita borde y fondo a un marco de mapa o leyenda."""
    try:
        cim = elm.getDefinition("V3")
        if getattr(cim, "graphicFrame", None):
            cim.graphicFrame.borderSymbol = None
            cim.graphicFrame.backgroundSymbol = None
        if hasattr(cim, "showTitle"):
            cim.showTitle = False
        elm.setDefinition(cim)
    except Exception as e:
        print(f"  (no pude quitar el marco: {e})")
 
 
def texto(lyt, txt, size, estilo_fuente="Regular", nombre=None):
    return aprx.createTextElement(lyt, arcpy.Point(1, 1), "POINT", txt, size,
                                  FUENTE_LETRA, estilo_fuente, name=nombre)
 
 
def capas_superiores(m):
    return [l for l in m.listLayers() if l.longName == l.name]
 
 
def buscar(m, nombre):
    for l in capas_superiores(m):
        if norm(l.name) == norm(nombre):
            return l
    return None
 
 
def preparar_mapa(nombre_copia, capa_tema):
    for viejo in aprx.listMaps(nombre_copia):
        aprx.deleteItem(viejo)
    m = aprx.copyItem(origen, nombre_copia)
    base = {norm(b) for b in CAPAS_BASE}
    for l in capas_superiores(m):
        if MAPA_BASE and norm(l.name) in base:
            m.removeLayer(l)            # se reemplaza por el mapa base nuevo
        else:
            l.visible = norm(l.name) in base or norm(l.name) == norm(capa_tema)
    if MAPA_BASE:
        try:
            m.addBasemap(MAPA_BASE)
        except Exception as e:
            print(f"  (no pude poner el mapa base '{MAPA_BASE}': {e})")
    return m
 
 
def encuadre(mf, capa, ancho, alto):
    """Deja la ciudad a la derecha del centro, para que el título y la
    leyenda queden sobre el mar y la tierra vacía de la izquierda."""
    e = mf.getLayerExtent(capa, False, True)
    ew, eh = e.XMax - e.XMin, e.YMax - e.YMin
    aspecto = ancho / alto
    h = eh * 1.12
    w = h * aspecto
    if w < ew * 1.6:
        w = ew * 1.6
        h = w / aspecto
    cx, cy = (e.XMin + e.XMax) / 2, (e.YMin + e.YMax) / 2
    xmin = cx - 0.62 * w
    ymin = cy - 0.5 * h
    mf.camera.setExtent(arcpy.Extent(xmin, ymin, xmin + w, ymin + h,
                                     spatial_reference=e.spatialReference))
 
 
ST_NORTE = estilo("NORTH_ARROW", ["ArcGIS North 3", "ArcGIS North 1"])
ST_ESCALA = estilo("SCALE_BAR", ["Scale Line 1 Metric", "Alternating Scale Bar 1 Metric"])
ST_LEYENDA = estilo("LEGEND", ["Legend 1"])
anio = datetime.date.today().strftime("%Y")
M = 0.15                     # margen blanco alrededor del mapa (pulgadas)
W, H = ANCHO, ALTO
 
for capa_tema, titulo, subtitulo, archivo in TEMAS:
    if not buscar(origen, capa_tema):
        print(f"⚠ No encontré la capa '{capa_tema}' en '{MAPA_ORIGEN}'. La salto.")
        continue
 
    nombre_titulo = titulo.replace("\n", " ").title()
    m = preparar_mapa(f"Mapa - {nombre_titulo}", capa_tema)
    tema = buscar(m, capa_tema)
 
    nombre_lyt = f"Layout - {nombre_titulo}"
    for viejo in aprx.listLayouts(nombre_lyt):
        aprx.deleteItem(viejo)
    lyt = aprx.createLayout(W, H, "INCH", nombre_lyt)
 
    # --- Mapa a toda página ---
    mf = lyt.createMapFrame(arcpy.Extent(M, M, W - M, H - M), m, "Mapa")
    sin_marco(mf)
    try:
        encuadre(mf, tema, W - 2 * M, H - 2 * M)
    except Exception as e:
        print(f"  (no pude encuadrar, uso zoom simple: {e})")
        mf.camera.setExtent(mf.getLayerExtent(tema, False, True))
 
    # --- Título grande arriba a la izquierda, sobre el mar ---
    t = texto(lyt, titulo, 44, "Bold", "Titulo")
    colorear(t, COLOR_TITULO, COLOR_HALO, 1.5)
    colocar(t, 0.55, H - 0.45)
 
    s = texto(lyt, subtitulo, 16, "Semibold", "Subtitulo")
    colorear(s, COLOR_TITULO, COLOR_HALO, 1.0)
    colocar(s, 0.6, H - 0.45 - 1.65)
 
    # --- Leyenda flotante abajo a la izquierda, sin marco ni título ---
    if ST_LEYENDA:
        ley = lyt.createMapSurroundElement(arcpy.Point(1, 1), "LEGEND", mf, ST_LEYENDA, "Leyenda")
        try:
            for item in list(ley.items):
                if norm(item.name) != norm(tema.name):
                    ley.removeItem(item)
        except Exception:
            pass
        sin_marco(ley)
        try:
            ley.fittingStrategy = "AdjustFontSize"
        except Exception:
            pass
        colocar(ley, 0.55, 0.5, ancla="BottomLeftCorner", w=3.4, h=2.6)
 
    # --- Norte arriba a la derecha, escala y fuentes abajo a la derecha ---
    if MOSTRAR_NORTE and ST_NORTE:
        n = lyt.createMapSurroundElement(arcpy.Point(1, 1), "NORTH_ARROW", mf, ST_NORTE, "Norte")
        colocar(n, W - 0.55, H - 0.45, ancla="TopRightCorner", h=0.55)
    if MOSTRAR_ESCALA and ST_ESCALA:
        esc = lyt.createMapSurroundElement(arcpy.Point(1, 1), "SCALE_BAR", mf, ST_ESCALA, "Escala")
        colocar(esc, W - 0.55, 1.05, ancla="BottomRightCorner", w=2.0)
 
    sr = m.spatialReference.name.replace("_", " ") if m.spatialReference else ""
    f = texto(lyt, f"{FUENTES}\nCoordinate system: {sr} · Map by {AUTOR}, {anio}", 8, nombre="Fuentes")
    colorear(f, COLOR_FUENTES, alineacion="Right")
    colocar(f, W - 0.55, 0.4, ancla="BottomRightCorner")
 
    salida = os.path.join(OUT_DIR, f"{archivo}.png")
    lyt.exportToPNG(salida, resolution=300)
    print(f"✓ {nombre_titulo} -> {salida}")
 
aprx.save()
print("Listo. Los layouts quedan en Catalog > Layouts por si quieres mover algo a mano.")

Guardando imágenes en: D:\ProyectoCroma\Proyecto Barranquilla\Catastro Barranquilla\mapas_presentacion
✓ Land Value Barranquilla -> D:\ProyectoCroma\Proyecto Barranquilla\Catastro Barranquilla\mapas_presentacion\mapa1.png
✓ Socioeconomic Stratum -> D:\ProyectoCroma\Proyecto Barranquilla\Catastro Barranquilla\mapas_presentacion\mapa2.png
✓ Residential Land Use -> D:\ProyectoCroma\Proyecto Barranquilla\Catastro Barranquilla\mapas_presentacion\mapa3.png
✓ Property Density -> D:\ProyectoCroma\Proyecto Barranquilla\Catastro Barranquilla\mapas_presentacion\mapa4.png
✓ Land Value × Stratum -> D:\ProyectoCroma\Proyecto Barranquilla\Catastro Barranquilla\mapas_presentacion\mapa5.png
✓ Model Residuals -> D:\ProyectoCroma\Proyecto Barranquilla\Catastro Barranquilla\mapas_presentacion\mapa6.png
Listo. Los layouts quedan en Catalog > Layouts por si quieres mover algo a mano.
